In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 04 — Track B: Phase 2 Hierarchical Efficient Fine-Tuning (HEFT)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from src.utils import load_config\n",
    "from src.data import load_rdiversevul, load_holdout_split\n",
    "\n",
    "cfg_b = load_config('configs/track_b.yaml')\n",
    "cfg_heft = load_config('configs/heft.yaml')\n",
    "\n",
    "df = load_rdiversevul()\n",
    "train_idx, _ = load_holdout_split()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from src.track_b.model import build_model\n",
    "from src.track_b.lora import apply_lora, freeze_lora, load_lora_adapter\n",
    "from src.track_b.reft import apply_reft\n",
    "from src.track_b.train import run_track_b_experiment\n",
    "\n",
    "BEST_LORA_CHECKPOINT_PATH = 'results/checkpoints/best_exp3_lora'\n",
    "OPTIMAL_LORA_RANK = 16\n",
    "\n",
    "for reft_rank in cfg_heft.reft.r:\n",
    "    model = build_model(cfg_b)\n",
    "    model = apply_lora(model, cfg_heft, target_r=OPTIMAL_LORA_RANK)\n",
    "    load_lora_adapter(model, BEST_LORA_CHECKPOINT_PATH)\n",
    "    freeze_lora(model)\n",
    "    \n",
    "    heft_model = apply_reft(model, cfg_heft, target_reft_r=reft_rank)\n",
    "    run_track_b_experiment(exp_id=f'EXP_4_reft_r{reft_rank}', cfg=cfg_b, heft_cfg=cfg_heft, df=df, train_idx=train_idx)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "cfg_b.model.name = 'answerdotai/ModernBERT-base'\n",
    "\n",
    "for reft_rank in cfg_heft.reft.r:\n",
    "    model = build_model(cfg_b)\n",
    "    heft_model = apply_reft(model, cfg_heft, target_reft_r=reft_rank)\n",
    "    run_track_b_experiment(exp_id=f'EXP_5_modernbert_reft_r{reft_rank}', cfg=cfg_b, heft_cfg=cfg_heft, df=df, train_idx=train_idx)"
   ]
  }
 ],
 "metadata": {
  "kernelspec": { "display_name": "Python 3", "language": "python", "name": "python3" },
  "language_info": { "name": "python", "version": "3.10.0" }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}